# Customer Churn Analysis

Using `cleaned_dataset.csv` to define the business question, find insights related to churn, and make a recommendation.

In [1]:
import pandas as pd

df = pd.read_csv("cleaned_dataset.csv")
df.head()

,CustomerID,Age,State,Income,Purchases,LastPurchaseDate,Review,Churn,LastPurchaseDate_missing,PositiveReview
0,CUST001,40.0,California,40000.0,0.0,NaN,No Review,No,True,0
1,CUST002,40.0,California,40000.0,1.0,NaN,No Review,No,True,0
2,CUST005,45.0,California,40000.0,2.0,2021-05-20,Loved it!!!,Yes,False,1
3,CUST006,45.0,New York,1000000.0,3.0,NaN,terrible service...,Yes,True,0
4,CUST007,45.0,New York,60000.0,2.0,NaN,Ok,Yes,True,0


## Part 1 — Define the Business Question

**What are we ultimately trying to predict?**

We are trying to predict `Churn` — whether a given customer will stop doing business with the company (`Churn = Yes`) or remain an active customer (`Churn = No`).

**If we could identify customers at higher risk of churn, what could the business do differently?**

The business could proactively target those customers with retention efforts before they leave, rather than reacting after the fact. Examples include personalized outreach or check-ins, loyalty discounts or promotions, prioritized customer support, or win-back offers. Focusing retention spend on the customers most likely to churn (instead of spreading it evenly across the whole customer base) would make retention efforts more efficient and cost-effective.

## Part 2 — Find Two Business Insights

In [2]:
# Calculate the overall churn rate
overall_churn_rate = (df["Churn"] == "Yes").mean()
print(f"Overall churn rate: {overall_churn_rate:.2%}")

Overall churn rate: 38.24%


In [3]:
# Compare the churn rate for PositiveReview = 1 vs PositiveReview = 0
churn_by_review = df.groupby("PositiveReview")["Churn"].apply(lambda s: (s == "Yes").mean())
churn_by_review.rename("churn_rate")

PositiveReview
0    0.309524
1    0.500000
Name: churn_rate, dtype: float64

In [4]:
# Additional variable: Purchases — compare average purchases for churned vs. retained customers
avg_purchases_by_churn = df.groupby("Churn")["Purchases"].mean()
avg_purchases_by_churn.rename("avg_purchases")

Churn
No     2.333333
Yes    2.961538
Name: avg_purchases, dtype: float64

### Other variables compared to churn

The assignment only requires one additional variable, but it is useful to scan a few more (Income, Age, State, and recency of last purchase) to see which look most worth following up on.

In [5]:
# Income vs. churn
avg_income_by_churn = df.groupby("Churn")["Income"].mean()
avg_income_by_churn.rename("avg_income")

Churn
No     279761.904762
Yes    199615.384615
Name: avg_income, dtype: float64

In [6]:
# Age vs. churn
avg_age_by_churn = df.groupby("Churn")["Age"].mean()
avg_age_by_churn.rename("avg_age")

Churn
No     41.785714
Yes    42.500000
Name: avg_age, dtype: float64

In [7]:
# State vs. churn
churn_by_state = df.groupby("State")["Churn"].apply(lambda s: (s == "Yes").mean())
churn_by_state.rename("churn_rate")

State
California    0.454545
New York      0.346154
Unknown       0.222222
Name: churn_rate, dtype: float64

In [8]:
# Recency of last purchase vs. churn (LastPurchaseDate_missing = no purchase date on file)
churn_by_recency = df.groupby("LastPurchaseDate_missing")["Churn"].apply(lambda s: (s == "Yes").mean())
churn_by_recency.rename("churn_rate")

LastPurchaseDate_missing
False    0.361702
True     0.428571
Name: churn_rate, dtype: float64

## Part 3 — Make a Recommendation

The most useful insight is that customers who left a positive review actually churn at a *higher* rate (50%) than customers without a positive review (about 31%), which is the opposite of what we'd normally expect and shows that review sentiment alone is not a reliable churn signal here. Management could use this to dig deeper into why satisfied-looking customers are still leaving — for example, checking whether positive reviewers tend to be one-time high-value purchasers or whether some other factor (timing, product category, price change) is driving both the review and the departure — rather than assuming a good review means a safe customer. This relationship does not prove that leaving a positive review causes churn; with only 68 customers and no control for other variables like income, purchase count, or timing, the pattern could easily reflect a confounding factor or random noise rather than a causal link. Confirming causation would require a more controlled comparison, such as tracking churn for otherwise-similar customer segments over time or running an experiment, rather than relying on a single cross-sectional comparison like this one.